In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

plt.style.use('seaborn-v0_8-whitegrid')


def find_repo_root() -> Path:
    current = Path.cwd().resolve()
    while current != current.parent:
        if (current / 'src' / 'models' / 'adapter.py').exists():
            return current
        current = current.parent
    raise RuntimeError('Could not locate DriftDetect repo root from the current working directory.')


def curve_stats(values: np.ndarray) -> dict[str, np.ndarray]:
    mean = values.mean(axis=0)
    std = values.std(axis=0, ddof=1)
    ci = 1.96 * std / np.sqrt(values.shape[0])
    return {'mean': mean, 'std': std, 'ci': ci}


REPO_ROOT = find_repo_root()
RESULTS_PATH = REPO_ROOT / 'results' / 'tables' / 'decoder_d3_results.npz'
FIGURE_DIR = REPO_ROOT / 'results' / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

PLOT_START = 10
PLOT_END = 189
PLOT_SLICE = slice(PLOT_START, PLOT_END + 1)
SUMMARY_STEPS = [25, 50, 100, 150, 175]

with np.load(RESULTS_PATH) as data:
    nonlinearity_gap = data['nonlinearity_gap'].astype(np.float64)
    actual_obs_diff_norm = data['actual_obs_diff_norm'].astype(np.float64)
    ratio = data['ratio'].astype(np.float64)
    seeds = data['seeds']

n_seeds, horizon = nonlinearity_gap.shape
steps = np.arange(horizon)
steps_plot = steps[PLOT_SLICE]

gap_stats = curve_stats(nonlinearity_gap)
actual_stats = curve_stats(actual_obs_diff_norm)
ratio_stats = curve_stats(ratio)
overall_mean_ratio = float(ratio[:, PLOT_SLICE].mean())

print(f'Loaded {RESULTS_PATH.relative_to(REPO_ROOT)}')
print(f'nonlinearity_gap shape: {nonlinearity_gap.shape}')
print(f'actual_obs_diff_norm shape: {actual_obs_diff_norm.shape}')
print(f'ratio shape: {ratio.shape}')
print(f'seeds: {seeds[0]}-{seeds[-1]}')
print(f'overall mean ratio [{PLOT_START}, {PLOT_END}]: {overall_mean_ratio:.3f}')


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 14), dpi=300, sharex=True)
fig.suptitle('D3: Linear vs Nonlinear Separation', fontsize=15, y=0.995)

panels = [
    {
        'ax': axes[0],
        'stats': gap_stats,
        'color': '#dc2626',
        'ylabel': 'Nonlinearity Gap L2',
        'title': '(a) Nonlinearity gap',
    },
    {
        'ax': axes[1],
        'stats': actual_stats,
        'color': '#2563eb',
        'ylabel': 'Actual Obs Difference L2',
        'title': '(b) Actual decoded obs difference',
    },
    {
        'ax': axes[2],
        'stats': ratio_stats,
        'color': '#ea580c',
        'ylabel': 'Gap / Actual Obs Difference',
        'title': '(c) Nonlinearity ratio',
    },
]

for panel in panels:
    ax = panel['ax']
    mean = panel['stats']['mean'][PLOT_SLICE]
    ci = panel['stats']['ci'][PLOT_SLICE]
    color = panel['color']
    ax.plot(steps_plot, mean, color=color, linewidth=2.0)
    ax.fill_between(
        steps_plot,
        mean - ci,
        mean + ci,
        color=color,
        alpha=0.20,
        linewidth=0,
    )
    ax.set_xlim(PLOT_START, PLOT_END)
    ax.set_ylabel(panel['ylabel'])
    ax.set_title(panel['title'], loc='left', fontsize=12)
    ax.grid(True, alpha=0.25, linewidth=0.6)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

axes[2].axhline(1.0, color='#111827', linestyle='--', linewidth=1.1, label='ratio = 1.0')
axes[2].axhline(0.5, color='#6b7280', linestyle='--', linewidth=1.0, label='moderate nonlinearity')
axes[2].axhline(0.1, color='#9ca3af', linestyle='--', linewidth=1.0, label='approximately linear')
axes[2].legend(frameon=True, loc='best')
axes[2].set_xlabel('Imagination Horizon Step')

fig.tight_layout()
three_panel_path = FIGURE_DIR / 'd3_linear_nonlinear.pdf'
fig.savefig(three_panel_path, bbox_inches='tight')
plt.show()
print(f'Saved figure: {three_panel_path.relative_to(REPO_ROOT)}')


In [ ]:
scatter_x = actual_obs_diff_norm.reshape(-1)
scatter_y = nonlinearity_gap.reshape(-1)

rng = np.random.default_rng(0)
max_points = 2000
if scatter_x.size > max_points:
    indices = rng.choice(scatter_x.size, size=max_points, replace=False)
    scatter_x_plot = scatter_x[indices]
    scatter_y_plot = scatter_y[indices]
else:
    scatter_x_plot = scatter_x
    scatter_y_plot = scatter_y

fig, ax = plt.subplots(figsize=(8, 6), dpi=300)
ax.scatter(scatter_x_plot, scatter_y_plot, s=12, alpha=0.35, color='#4b5563', edgecolors='none')

axis_max = float(max(scatter_x.max(), scatter_y.max()))
ax.plot([0, axis_max], [0, axis_max], color='#dc2626', linestyle='--', linewidth=1.2, label='ratio = 1.0')

ax.set_xlim(0, axis_max * 1.02)
ax.set_ylim(0, axis_max * 1.02)
ax.set_xlabel('Actual Obs Difference L2')
ax.set_ylabel('Nonlinearity Gap L2')
ax.set_title('Nonlinearity Gap vs Actual Obs Difference')
ax.legend(frameon=True)
ax.grid(True, alpha=0.25, linewidth=0.6)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

fig.tight_layout()
scatter_path = FIGURE_DIR / 'd3_scatter.pdf'
fig.savefig(scatter_path, bbox_inches='tight')
plt.show()
print(f'Saved figure: {scatter_path.relative_to(REPO_ROOT)}')


In [ ]:
print('D3 summary statistics across seeds')
print()
print(
    f"{'Step':>6}  {'Gap mean +/- std':>22}  "
    f"{'Actual mean +/- std':>24}  {'Ratio mean +/- std':>23}"
)
for step in SUMMARY_STEPS:
    print(
        f"{step:>6}  "
        f"{gap_stats['mean'][step]:>8.3f} +/- {gap_stats['std'][step]:<8.3f}  "
        f"{actual_stats['mean'][step]:>8.3f} +/- {actual_stats['std'][step]:<8.3f}  "
        f"{ratio_stats['mean'][step]:>8.3f} +/- {ratio_stats['std'][step]:<8.3f}"
    )

print()
print(f'Overall mean ratio across steps [{PLOT_START}, {PLOT_END}]: {overall_mean_ratio:.3f}')


## D3 Result Summary

- The nonlinearity gap is approximately equal to the actual observation difference, with ratio near 1.0.
- This means `decode(imag - true)` produces outputs very different from `decode(imag) - decode(true)`.
- However, this does **not** contradict D2's finding that the decoder is locally linear.
- Explanation: `decode(latent_delta)` feeds a difference vector to the decoder, which is far out-of-distribution. The decoder's output on this OOD input is essentially meaningless, so the gap is large by construction.
- The combined D2 + D3 picture: the decoder is locally linear near the training distribution but behaves unpredictably on OOD inputs.
- This explains D1: the trained decoder amplifies drift because imagined latents drift OOD over long horizons, and the trained decoder's learned mapping becomes unreliable in those OOD regions. A random decoder has no in-distribution vs OOD distinction, so it shows less amplification.
- Conclusion for Finding E: the original decoder amplification hypothesis is partially supported by D1, but the mechanism is OOD instability rather than directional amplification of `dc_trend`; D2 refuted the directional-bias version. The decoder story is a supporting detail, not the main driver of trend dominance.
